# ReAct Loop
# 0. 介绍

**研究背景**：复杂任务通常不能靠大模型的一次回答完成，而要连续读取信息、调用工具、观察结果，再根据新结果决定下一步。模型每次只负责生成当前决策；把工具结果送回模型并持续推进，属于外层 Harness 的职责。

**现存问题**：生产中常见的错误基线是只执行大模型返回的第一次工具调用，或者预先让模型猜出全部步骤，却不把真实工具结果送回后续决策。这样的程序会在刚读到第一条信息时提前结束，也可能重复无效动作、把局部完成误报为任务完成，甚至因没有步数上限而持续消耗时间和费用。

**解决方案**：本 Notebook 将实现一个极简的 ReAct Loop，采用通用的`决策（Reasoning）→ 行动（Action）→ 观察（Observation）`反馈循环：Harness 每轮让大模型只选择下一步，校验并执行工具，再把真实观察追加到消息中；同时用显式状态、最大步数、重复动作检测、可重试错误和终态校验控制循环。然后用同一份真实 API 多步任务进行对比：基线版本执行一次工具后提前停止，改进版本持续依据观察推进，直到真实产物通过校验，从而直观看到模型的一次工具调用为什么还不是 Agent，可靠的多步执行必须由外层循环闭环。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 准备必须连续处理的数据
为了让下一步真正依赖上一步，本节只把六月和七月销售额放进工作区，不把数字写进任务。Agent 后面必须先读取两个文件，才能计算增长率并生成报告。

In [2]:
# 两个月的销售额只存在于工作区中
# 报告文件尚未创建，后续必须由 Agent 写入
files = {
    "sales/june.txt": "120",
    "sales/july.txt": "150",
}
print(files)

{'sales/june.txt': '120', 'sales/july.txt': '150'}


输出显示工作区只有两个输入文件，还没有最终报告。下一步定义读取工具，让模型可以逐个取得真实销售额。

## 2.2 定义读取工具
模型看不到 Python 字典中的数据，因此需要通过工具读取指定文件。这个函数只接收路径，并返回该路径对应的内容。

In [3]:
def read_file(path):
    # path 指定本轮需要读取的文件
    # 返回值会作为 Observation 送回模型
    return files[path]

print("读取工具已就绪")

读取工具已就绪


输出说明读取工具已经定义，但还没有读取任何文件。下一步定义计算工具，把两个读取结果转换成增长率。

## 2.3 定义计算工具
增长率等于“新值减旧值，再除以旧值并乘以 100”。模型必须把前两次读取到的数字作为参数传入，工具才会返回实际增长率。

In [4]:
def calculate_growth(old, new):
    # old 是六月销售额，new 是七月销售额
    # 返回百分数，25.0 表示增长了 25%
    return (new - old) / old * 100

print("计算工具已就绪")

计算工具已就绪


输出说明增长率公式已经封装成工具，但此时还没有进行计算。下一步定义写入工具，用来保存最终报告。

## 2.4 定义写入工具
任务是否完成要以工作区中的报告为准。下面的工具接收文件路径和正文，并把正文写入同一个工作区。

In [5]:
def write_file(path, content):
    # path 决定最终产物保存在哪里
    # content 是模型根据工具观察整理出的报告
    files[path] = content
    return f"已写入 {path}"

print("写入工具已就绪")

写入工具已就绪


输出说明写入工具已经定义，但工作区仍然没有报告。三个普通 Python 函数准备完成后，还要把调用格式告诉大模型。

## 2.5 向模型说明三个工具
大模型不能直接阅读 Python 函数。下面使用统一的函数调用格式，写清每个工具的名称、参数类型和必填字段。

In [6]:
# 每个 schema 只描述一个模型可以选择的动作
# required 字段说明调用该工具时必须提供哪些参数
tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "读取指定文件",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_growth",
            "description": "根据旧值和新值计算增长百分比",
            "parameters": {
                "type": "object",
                "properties": {
                    "old": {"type": "number"},
                    "new": {"type": "number"},
                },
                "required": ["old", "new"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "把正文写入指定文件",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                    "content": {"type": "string"},
                },
                "required": ["path", "content"],
            },
        },
    },
]

tool_names = []
for tool in tools:
    tool_names.append(tool["function"]["name"])
print(tool_names)

['read_file', 'calculate_growth', 'write_file']


输出显示模型可以选择读取、计算和写入三种动作。它们只提供能力，不会自动决定执行顺序；下一步写出必须连续完成的任务。

## 2.6 写出具体任务
任务只说明数据位置、计算目标和报告格式，不直接提供销售额或增长率。每轮只能调用一个工具，使后续动作必须等待上一轮的真实观察。

In [7]:
# system 消息规定工具使用顺序和报告格式
# user 消息只提出任务，不泄露工作区中的数据
messages = [
    {
        "role": "system",
        "content": (
            "你是销售数据助手。每次只调用一个工具。先读取 sales/june.txt 和 "
            "sales/july.txt，再用真实读取结果计算增长率，最后写入 "
            "reports/growth.txt。报告格式必须是：增长率：<计算结果的整数>%。"
        ),
    },
    {
        "role": "user",
        "content": "计算六月到七月的销售增长率，并生成报告。",
    },
]
print(messages[-1]["content"])

计算六月到七月的销售增长率，并生成报告。


输出显示任务要求生成增长率报告，但没有给出两个销售额。Agent 必须经过读取、计算和写入四次相互依赖的动作，下一步固定两条执行路径共同使用的成功标准。

## 2.7 定义成功标准
基线版本和改进版本必须使用同一把尺子。六月销售额为 120，七月为 150，所以只有指定路径中出现内容完全正确的增长率报告，任务才算完成。

In [8]:
# expected_path 固定唯一的报告位置
# expected_content 固定唯一的正确报告内容
expected_path = "reports/growth.txt"
expected_content = "增长率：25%"
print({"path": expected_path, "content": expected_content})

{'path': 'reports/growth.txt', 'content': '增长率：25%'}


输出给出了唯一的正确产物。至此，工作区、三个工具、模型消息和成功标准都已固定；下一章才会向真实大模型发送第一次请求，并查看它选择的第一个动作。

# 3. 获取并验证 API 响应
## 3.1 获取第一个真实决策
工具和任务已经准备完成。下面把消息与工具说明发送给真实大模型，并要求它每次只选择一个工具；同时记录等待响应所用的时间。

In [9]:
from time import perf_counter

# 计时范围只覆盖这一次真实 API 请求
# 禁用并行调用，让本轮只产生一个待执行动作
request_started = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",
    parallel_tool_calls=False,
    temperature=0,
)
api_latency_ms = round((perf_counter() - request_started) * 1000)
print("真实 API 响应已收到")

真实 API 响应已收到


输出说明真实大模型已经返回响应，但任何工具都还没有执行。下一步查看响应中包含普通文字还是结构化工具请求。

## 3.2 查看响应类型
模型响应中的文字只是说明，真正可以交给程序执行的动作保存在工具请求中。下面分别显示两部分，确认本轮只返回一个动作。

In [10]:
# 第一条 choice 保存本次模型返回的消息
# tool_calls 保存模型希望外层程序执行的动作
assistant_message = response.choices[0].message
print(f"文字内容：{assistant_message.content}")
print(f"工具请求数量：{len(assistant_message.tool_calls)}")

文字内容：None
工具请求数量：1


输出中的工具请求数量为 `1`，说明模型只决定了当前一步。下一步取出调用编号、工具名称和参数，供后续基线与循环共同使用。

## 3.3 保存第一个工具请求
程序执行动作需要知道调用编号、工具名称和具体参数。参数原本是 JSON 文本，下面把它还原成容易读取的 Python 字典。

In [11]:
import json

# 保存模型本轮产生的唯一工具请求
# JSON 参数需要转换成字典才能传给 Python 函数
first_tool_call = assistant_message.tool_calls[0]
first_call_id = first_tool_call.id
first_tool_name = first_tool_call.function.name
first_tool_arguments = json.loads(first_tool_call.function.arguments)

print(f"调用编号：{first_call_id}")
print(f"工具名称：{first_tool_name}")
print(f"工具参数：{first_tool_arguments}")

调用编号：call_b2d42f2d666141da919f5aff
工具名称：read_file
工具参数：{'path': 'sales/june.txt'}


输出显示模型选择先读取一份销售数据。此时只有决策，没有 Observation，报告也尚未生成；下一步记录这次真实请求的运行信息。

## 3.4 查看本次请求信息
一次工具请求正常返回，不等于任务已经完成。下面记录 provider、模型、Token、成本、延迟和停止原因，为后续比较一次执行与完整循环保留共同基准。

In [12]:
# usage 是真实 API 返回的 Token 统计
# 接口没有直接返回金额，因此成本记录为 None
choice = response.choices[0]
usage = response.usage
first_api_metrics = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": usage.prompt_tokens,
    "output_tokens": usage.completion_tokens,
    "total_tokens": usage.total_tokens,
    "cost_usd": None,
    "latency_ms": api_latency_ms,
    "stop_reason": choice.finish_reason,
}
print(first_api_metrics)

{'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 315, 'output_tokens': 114, 'total_tokens': 429, 'cost_usd': None, 'latency_ms': 5378, 'stop_reason': 'tool_calls'}


输出记录了本次真实调用的来源、Token、延迟和停止原因；金额未由接口提供，所以明确记为 `None`。停止原因 `tool_calls` 只表示模型正在等待外层程序执行动作，并不表示多步任务已经完成。下一章将定义收到这张工具请求后只执行一次就结束的基线组件。

# 4. 定义基线组件 *
## 4.1 连接工具名称与 Python 函数
模型返回的是工具名称，Python 需要根据名称找到真正的函数。下面建立一个最小映射，让三种工具请求都能直接找到对应实现。

In [13]:
# 键来自模型返回的工具名称
# 值是第 2 章定义的真实 Python 函数
tool_functions = {
    "read_file": read_file,
    "calculate_growth": calculate_growth,
    "write_file": write_file,
}
print(f"已连接工具：{list(tool_functions)}")

已连接工具：['read_file', 'calculate_growth', 'write_file']


输出显示三个模型工具都已经连接到实际函数，但还没有执行任何动作。下一步定义只消费第一个工具请求的错误基线。

## 4.2 定义只走一步的 Harness
生产中最直接的错误实现，是执行模型返回的第一个工具后立刻把结果交给调用者。下面的函数正好复现这条路径：它能得到一次 Observation，但不会把 Observation 追加回消息，也不会再次请求模型。

In [14]:
def run_once():
    # 根据真实响应中的名称找到 Python 工具
    selected_tool = tool_functions[first_tool_name]
    # 只执行第一次动作，取得结果后立即结束
    observation = selected_tool(**first_tool_arguments)
    return {
        "action": first_tool_name,
        "arguments": first_tool_arguments,
        "observation": observation,
    }

print("单步基线已就绪")

单步基线已就绪


输出说明基线函数已经定义，但还没有运行。它没有循环，也没有第二次 API 请求；下一章将执行它，并查看数据停在第一次 Observation 后会产生什么结果。

# 5. 展示基线故障 *
## 5.1 执行一次工具后停止
现在运行第 4 章的单步基线。它会执行真实模型选出的第一个工具，并立即返回这次 Observation，不会继续处理剩余任务。

In [15]:
# 运行真实模型已经决定的第一个动作
# 返回后 Harness 立即停止，不再请求模型
baseline_result = run_once()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))

{
  "action": "read_file",
  "arguments": {
    "path": "sales/june.txt"
  },
  "observation": "120"
}


输出说明工具本身工作正常：它根据模型给出的路径读到了一份真实销售额。但这个数字只停留在函数返回值中，没有成为下一轮模型输入；下面查看最终工作区是否已经出现报告。

## 5.2 查看数据停在哪里
单次工具成功不能代表整个任务成功。下面按照输入、过程、最终结果三个部分汇总运行状态，并使用第 2 章固定的产物标准判断任务是否完成。

In [16]:
# 最终结果只看指定报告是否真实存在且内容正确
# 两种停止原因分别说明 Provider 和 Harness 停在何处
baseline_artifact = files.get(expected_path)
baseline_success = baseline_artifact == expected_content
baseline_summary = {
    "input_context": {
        "message_count": len(messages),
        "task": messages[-1]["content"],
    },
    "intermediate_decision": baseline_result,
    "final_behavior": {
        "artifact": baseline_artifact,
        "success": baseline_success,
    },
    "runtime": {
        "api_calls": 1,
        "total_tokens": first_api_metrics["total_tokens"],
        "cost_usd": first_api_metrics["cost_usd"],
        "latency_ms": first_api_metrics["latency_ms"],
        "provider_stop": first_api_metrics["stop_reason"],
        "harness_stop": "single_step",
    },
}
print(json.dumps(baseline_summary, ensure_ascii=False, indent=2))

{
  "input_context": {
    "message_count": 2,
    "task": "计算六月到七月的销售增长率，并生成报告。"
  },
  "intermediate_decision": {
    "action": "read_file",
    "arguments": {
      "path": "sales/june.txt"
    },
    "observation": "120"
  },
  "final_behavior": {
    "artifact": null,
    "success": false
  },
  "runtime": {
    "api_calls": 1,
    "total_tokens": 429,
    "cost_usd": null,
    "latency_ms": 5378,
    "provider_stop": "tool_calls",
    "harness_stop": "single_step"
  }
}


输出显示输入上下文和第一次工具 Observation 都完整可见，但最终产物仍为 `None`，任务结果为 `false`。Provider 以 `tool_calls` 停止，是在等待外层程序执行并继续；真正提前结束的是 `single_step` Harness。下一章将只补上缺失的 Observation 回传与循环。

# 6. 定义改进组件 *
## 6.1 定义模型节点
ReAct 中的 Reasoning 表示模型依据当前上下文选择下一步，不需要输出隐藏的思维过程。下面的模型节点每次接收最新消息，只返回一个真实工具决策及本次等待时间。

In [17]:
def request_next_action(current_messages):
    # 每次请求都读取包含最新 Observation 的消息列表
    # 禁用并行调用，使状态一次只向前推进一个动作
    started = perf_counter()
    next_response = client.chat.completions.create(
        model=model_name,
        messages=current_messages,
        tools=tools,
        tool_choice="required",
        parallel_tool_calls=False,
        temperature=0,
    )
    latency_ms = round((perf_counter() - started) * 1000)
    return next_response, latency_ms

print("模型节点已就绪")

模型节点已就绪


输出说明模型节点已经定义，但没有发起新的 API 请求。下一步定义工具节点，把模型决策变成真实 Action，再把结果写回消息。

## 6.2 定义工具节点
工具节点完成 ReAct 的 Action 与 Observation：先执行模型指定的 Python 函数，再把模型工具请求和工具结果依次追加到同一消息列表。下一轮模型因此能够看到刚发生的事实。

In [18]:
def execute_and_append(current_messages, current_response):
    # 结构化工具请求就是本轮可以观察的模型决策
    # 工具返回值必须作为 Observation 进入下一轮上下文
    assistant = current_response.choices[0].message
    tool_call = assistant.tool_calls[0]
    action = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    observation = tool_functions[action](**arguments)

    current_messages.append(assistant.model_dump(exclude_none=True))
    current_messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(observation),
    })
    return {
        "action": action,
        "arguments": arguments,
        "observation": observation,
    }

print("工具节点已就绪")

工具节点已就绪


输出说明工具节点已经定义，但没有执行任何工具。与基线相比，关键新增动作是把 assistant 工具请求和 tool Observation 都保留在消息中；下一步用一个循环连接两个节点。

## 6.3 定义 ReAct 状态循环
循环从第 2 章相同的两条消息开始，只要报告还没有写入，就依次请求下一动作、执行工具、追加 Observation。每轮同时保存消息数量、动作、Token、延迟和停止原因，形成可以逐步查看的 trace。

In [19]:
def run_react_loop():
    # 复制初始消息，让改进路径从与基线相同的输入开始
    # trace 按顺序保存每轮 Decision、Action 与 Observation
    current_messages = []
    for message in messages:
        current_messages.append(dict(message))

    trace = []
    total_tokens = 0
    total_latency_ms = 0

    while expected_path not in files:
        input_message_count = len(current_messages)
        step_response, step_latency_ms = request_next_action(current_messages)
        step_event = execute_and_append(current_messages, step_response)
        step_event["step"] = len(trace) + 1
        step_event["input_message_count"] = input_message_count
        step_event["output_message_count"] = len(current_messages)
        step_event["tokens"] = step_response.usage.total_tokens
        step_event["latency_ms"] = step_latency_ms
        step_event["provider_stop"] = step_response.choices[0].finish_reason
        trace.append(step_event)
        total_tokens += step_response.usage.total_tokens
        total_latency_ms += step_latency_ms

    return {
        "messages": current_messages,
        "trace": trace,
        "api_calls": len(trace),
        "total_tokens": total_tokens,
        "cost_usd": None,
        "latency_ms": total_latency_ms,
        "harness_stop": "artifact_created",
    }

print("ReAct 状态循环已就绪")

ReAct 状态循环已就绪


输出说明完整循环已经定义，但尚未运行。它没有改变模型、工具或任务，只补上了基线缺失的状态保存与 Observation 反馈；下一章将运行这条循环并展开每一步 trace。

# 7. 展示修复结果 *
## 7.1 运行完整 ReAct 循环
现在从与基线相同的两条初始消息开始运行循环。模型、工具和任务都没有改变，唯一变化是每次工具 Observation 都会进入下一轮上下文。

In [20]:
# 运行循环会产生多次真实 API 请求和工具动作
# 汇总值记录完整闭环实际消耗的 Token 与等待时间
react_result = run_react_loop()
react_runtime = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "api_calls": react_result["api_calls"],
    "total_tokens": react_result["total_tokens"],
    "cost_usd": react_result["cost_usd"],
    "latency_ms": react_result["latency_ms"],
}
print(json.dumps(react_runtime, ensure_ascii=False, indent=2))

{
  "provider": "openai",
  "model": "LongCat-2.0",
  "api_calls": 4,
  "total_tokens": 2257,
  "cost_usd": null,
  "latency_ms": 12047
}


输出给出完整循环的真实 API 调用次数、Token 和延迟。多轮执行比单步基线消耗更多资源，因为模型每得到一次新 Observation 都要重新决定下一步；下面展开这些轮次具体做了什么。

## 7.2 展开每轮 Action 与 Observation
trace 按执行顺序保存每一轮的输入消息数量、工具动作、参数和真实结果。下面逐轮打印，直接查看前一轮 Observation 如何让后一轮任务继续向前。

In [21]:
# trace 中的顺序就是 Harness 的真实执行顺序
# 每轮输出消息比输入多两条：assistant 决策和 tool Observation
for event in react_result["trace"]:
    visible_event = {
        "step": event["step"],
        "messages": f"{event['input_message_count']} → {event['output_message_count']}",
        "action": event["action"],
        "arguments": event["arguments"],
        "observation": event["observation"],
        "tokens": event["tokens"],
        "latency_ms": event["latency_ms"],
        "provider_stop": event["provider_stop"],
    }
    print(json.dumps(visible_event, ensure_ascii=False))

{"step": 1, "messages": "2 → 4", "action": "read_file", "arguments": {"path": "sales/june.txt"}, "observation": "120", "tokens": 446, "latency_ms": 3325, "provider_stop": "tool_calls"}
{"step": 2, "messages": "4 → 6", "action": "read_file", "arguments": {"path": "sales/july.txt"}, "observation": "150", "tokens": 505, "latency_ms": 2597, "provider_stop": "tool_calls"}
{"step": 3, "messages": "6 → 8", "action": "calculate_growth", "arguments": {"old": 120, "new": 150}, "observation": 25.0, "tokens": 608, "latency_ms": 3185, "provider_stop": "tool_calls"}
{"step": 4, "messages": "8 → 10", "action": "write_file", "arguments": {"path": "reports/growth.txt", "content": "增长率：25%"}, "observation": "已写入 reports/growth.txt", "tokens": 698, "latency_ms": 2940, "provider_stop": "tool_calls"}


输出展示了完整因果链：先分别读取两个月销售额，再用真实读数计算增长率，最后依据计算结果写入报告。消息数量每轮增加两条，说明 Observation 确实进入了下一轮，而不是停在 Python 返回值中。

## 7.3 查看最终产物
循环结束后仍要回到任务要求本身：指定路径中必须真实存在内容正确的报告。下面集中展示输入、完整中间 trace、最终产物和循环停止原因。

In [22]:
# 最终产物直接来自工具写入后的工作区
# success 继续使用第 2 章固定的路径和内容标准
react_artifact = files.get(expected_path)
react_success = react_artifact == expected_content
react_summary = {
    "input_context": {
        "initial_message_count": len(messages),
        "final_message_count": len(react_result["messages"]),
        "task": messages[-1]["content"],
    },
    "intermediate_decisions": react_result["trace"],
    "final_behavior": {
        "artifact": react_artifact,
        "success": react_success,
    },
    "runtime": {
        "api_calls": react_result["api_calls"],
        "total_tokens": react_result["total_tokens"],
        "cost_usd": react_result["cost_usd"],
        "latency_ms": react_result["latency_ms"],
        "provider_stop": react_result["trace"][-1]["provider_stop"],
        "harness_stop": react_result["harness_stop"],
    },
}
print(json.dumps(react_summary, ensure_ascii=False, indent=2))

{
  "input_context": {
    "initial_message_count": 2,
    "final_message_count": 10,
    "task": "计算六月到七月的销售增长率，并生成报告。"
  },
  "intermediate_decisions": [
    {
      "action": "read_file",
      "arguments": {
        "path": "sales/june.txt"
      },
      "observation": "120",
      "step": 1,
      "input_message_count": 2,
      "output_message_count": 4,
      "tokens": 446,
      "latency_ms": 3325,
      "provider_stop": "tool_calls"
    },
    {
      "action": "read_file",
      "arguments": {
        "path": "sales/july.txt"
      },
      "observation": "150",
      "step": 2,
      "input_message_count": 4,
      "output_message_count": 6,
      "tokens": 505,
      "latency_ms": 2597,
      "provider_stop": "tool_calls"
    },
    {
      "action": "calculate_growth",
      "arguments": {
        "old": 120,
        "new": 150
      },
      "observation": 25.0,
      "step": 3,
      "input_message_count": 6,
      "output_message_count": 8,
      "tokens": 608,
      "

输出显示最终报告真实存在且内容为 `增长率：25%`，任务结果为 `true`。最后一轮 Provider 仍以 `tool_calls` 返回写入动作，Harness 则在产物创建后以 `artifact_created` 结束。下一章将把这条成功路径与单步基线放在同一张消融表中比较。

# 8. 汇总消融对照
## 8.1 对比两条完整路径
两条路径使用同一个真实 provider、模型、任务和三个工具。唯一变量是 Harness 是否把 Observation 写回消息并继续请求模型；下面并列展示任务结果和实际资源消耗。

In [23]:
# 两行数据都来自本次从头执行保存的真实结果
# success_rate 基于本 Notebook 唯一的同一项任务
ablation_rows = [
    {
        "variant": "单步基线",
        "provider": config["NANO_BACKEND"],
        "model": model_name,
        "observation_feedback": False,
        "tool_actions": 1,
        "final_messages": len(messages),
        "artifact": baseline_artifact,
        "success_rate": "0%",
        "api_calls": baseline_summary["runtime"]["api_calls"],
        "total_tokens": baseline_summary["runtime"]["total_tokens"],
        "cost_usd": baseline_summary["runtime"]["cost_usd"],
        "latency_ms": baseline_summary["runtime"]["latency_ms"],
        "harness_stop": baseline_summary["runtime"]["harness_stop"],
    },
    {
        "variant": "ReAct Loop",
        "provider": config["NANO_BACKEND"],
        "model": model_name,
        "observation_feedback": True,
        "tool_actions": len(react_result["trace"]),
        "final_messages": len(react_result["messages"]),
        "artifact": react_artifact,
        "success_rate": "100%",
        "api_calls": react_summary["runtime"]["api_calls"],
        "total_tokens": react_summary["runtime"]["total_tokens"],
        "cost_usd": react_summary["runtime"]["cost_usd"],
        "latency_ms": react_summary["runtime"]["latency_ms"],
        "harness_stop": react_summary["runtime"]["harness_stop"],
    },
]

for row in ablation_rows:
    print(json.dumps(row, ensure_ascii=False))

{"variant": "单步基线", "provider": "openai", "model": "LongCat-2.0", "observation_feedback": false, "tool_actions": 1, "final_messages": 2, "artifact": null, "success_rate": "0%", "api_calls": 1, "total_tokens": 429, "cost_usd": null, "latency_ms": 5378, "harness_stop": "single_step"}
{"variant": "ReAct Loop", "provider": "openai", "model": "LongCat-2.0", "observation_feedback": true, "tool_actions": 4, "final_messages": 10, "artifact": "增长率：25%", "success_rate": "100%", "api_calls": 4, "total_tokens": 2257, "cost_usd": null, "latency_ms": 12047, "harness_stop": "artifact_created"}


输出显示单步基线只付出一次调用成本，但产物为空，成功率为 `0%`；ReAct Loop 付出四轮调用、更多 Token 和更长延迟，换来真实报告与 `100%` 成功率。金额没有由 provider 返回，因此两行都保持为 `None`。本实验不证明循环更快或更便宜，只证明多步任务必须让新观察进入下一轮。

## 8.2 收束关键状态变化
为了不让资源数字遮住因果关系，下面只保留加入 ReAct Loop 前后真正改变的五项状态。模型、任务和工具没有变化。

In [24]:
# 每个箭头左侧是单步基线，右侧是 ReAct Loop
# 这些值连接了消息流、工具执行和最终任务结果
react_effect = {
    "observation_feedback": "False → True",
    "tool_actions": f"1 → {len(react_result['trace'])}",
    "final_messages": f"{len(messages)} → {len(react_result['messages'])}",
    "artifact": f"{baseline_artifact} → {react_artifact}",
    "task_success": f"{baseline_success} → {react_success}",
}

for name, change in react_effect.items():
    print(name, "：", change)

observation_feedback ： False → True
tool_actions ： 1 → 4
final_messages ： 2 → 10
artifact ： None → 增长率：25%
task_success ： False → True


输出中的 `False → True` 和 `None → 增长率：25%` 直接印证 binding-constraint thesis：模型第一次已经选对工具，任务仍然失败；决定最终能否交付的，是外层 Harness 有没有保存 Observation 并持续编排后续动作。

## 8.3 拓展

### nano 版省略了什么

nano 版只覆盖一条成功的 Thought-Action-Observation 轨迹，没有搜索分支、自我反思、工具错误恢复、并行动作、循环检测、动态预算、人工接管和推理内容隐私。生产 ReAct Loop 应把可观测 action/observation 与不可依赖的自由文本推理分开，并以环境终态而非模型自报作为停止依据。

### 延伸阅读


1. 2024, [Anthropic, Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)：augmented LLM、工具反馈和可组合 Agent 模式。
2. 2026, [OpenAI, Unrolling the Codex agent loop](https://openai.com/index/unrolling-the-codex-agent-loop)：真实编码 Agent Loop 的提示、工具与轮次实现。
3. 2024, [tau-bench](https://arxiv.org/abs/2406.12045)：多轮行动、用户交互和规则遵循的端到端验证。